In [1]:
import pandas as pd
from chembl_webresource_client.new_client import new_client

Search for target proteins

In [2]:
target = new_client.target
target_query = target.search("alzheimer's")
targets = pd.DataFrame.from_dict(target_query)
targets

,cross_references,organism,pref_name,score,species_group_flag,target_chembl_id,target_components,target_type,tax_id
0,[],Homo sapiens,Nucleosome-remodeling factor subunit BPTF,6.0,False,CHEMBL3085621,"[{'accession': 'Q12830', 'component_descriptio...",SINGLE PROTEIN,9606
1,[],Homo sapiens,Nicastrin,5.0,False,CHEMBL3418,"[{'accession': 'Q92542', 'component_descriptio...",SINGLE PROTEIN,9606
2,[],Homo sapiens,Gamma-secretase,5.0,False,CHEMBL2094135,"[{'accession': 'Q96BI3', 'component_descriptio...",PROTEIN COMPLEX,9606
3,[],Rattus norvegicus,Amyloid-beta precursor protein,4.0,False,CHEMBL3638365,"[{'accession': 'P08592', 'component_descriptio...",SINGLE PROTEIN,10116
4,[],Mus musculus,Amyloid-beta precursor protein,4.0,False,CHEMBL4523942,"[{'accession': 'P12023', 'component_descriptio...",SINGLE PROTEIN,10090
5,[],Homo sapiens,Amyloid-beta precursor protein,3.0,False,CHEMBL2487,"[{'accession': 'P05067', 'component_descriptio...",SINGLE PROTEIN,9606


Select and retrieve bioactivity data for Amyloid-beta precursor protein

In [3]:
selected_target = targets.target_chembl_id[5]
selected_target

'CHEMBL2487'

In [4]:
activity = new_client.activity
res = activity.filter(target_chembl_id=selected_target).filter(standard_type="IC50")

IC50 = Potency to elicit 50% inhibition of target protein

In [5]:
df = pd.DataFrame.from_dict(res)

In [6]:
df.head(3)

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,NaN,357577,[],CHEMBL678443,Inhibition of A-beta-42 production by inhibiti...,B,None,None,BAO_0000190,...,Homo sapiens,Amyloid-beta precursor protein,9606,None,None,IC50,uM,UO_0000065,NaN,5.0
1,None,NaN,357580,[],CHEMBL678443,Inhibition of A-beta-42 production by inhibiti...,B,None,None,BAO_0000190,...,Homo sapiens,Amyloid-beta precursor protein,9606,None,None,IC50,uM,UO_0000065,NaN,2.7
2,None,NaN,358965,[],CHEMBL678443,Inhibition of A-beta-42 production by inhibiti...,B,None,None,BAO_0000190,...,Homo sapiens,Amyloid-beta precursor protein,9606,None,None,IC50,uM,UO_0000065,NaN,1.8


In [7]:
# Save to csv
df.to_csv('data/bioactivity_data.csv', index=False)

In [8]:
! head bioactivity_data.csv

head: bioactivity_data.csv: No such file or directory


Handle missing data

In [9]:
df2 = df[df.standard_value.notna()]
df2

,action_type,activity_comment,activity_id,activity_properties,assay_chembl_id,assay_description,assay_type,assay_variant_accession,assay_variant_mutation,bao_endpoint,...,target_organism,target_pref_name,target_tax_id,text_value,toid,type,units,uo_units,upper_value,value
0,None,NaN,357577,[],CHEMBL678443,Inhibition of A-beta-42 production by inhibiti...,B,None,None,BAO_0000190,...,Homo sapiens,Amyloid-beta precursor protein,9606,None,None,IC50,uM,UO_0000065,NaN,5.0
1,None,NaN,357580,[],CHEMBL678443,Inhibition of A-beta-42 production by inhibiti...,B,None,None,BAO_0000190,...,Homo sapiens,Amyloid-beta precursor protein,9606,None,None,IC50,uM,UO_0000065,NaN,2.7
2,None,NaN,358965,[],CHEMBL678443,Inhibition of A-beta-42 production by inhibiti...,B,None,None,BAO_0000190,...,Homo sapiens,Amyloid-beta precursor protein,9606,None,None,IC50,uM,UO_0000065,NaN,1.8
3,None,NaN,368887,[],CHEMBL678443,Inhibition of A-beta-42 production by inhibiti...,B,None,None,BAO_0000190,...,Homo sapiens,Amyloid-beta precursor protein,9606,None,None,IC50,uM,UO_0000065,NaN,11.0
4,None,NaN,375954,[],CHEMBL678443,Inhibition of A-beta-42 production by inhibiti...,B,None,None,BAO_0000190,...,Homo sapiens,Amyloid-beta precursor protein,9606,None,None,IC50,uM,UO_0000065,NaN,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2065,None,1074589,28446179,[],CHEMBL5737142,BiaCore assay: β-sectretase inhibitor IV from ...,B,None,None,BAO_0000190,...,Homo sapiens,Amyloid-beta precursor protein,9606,None,None,IC50,nM,UO_0000065,NaN,520.0
2066,None,1074591,28446185,[],CHEMBL5737142,BiaCore assay: β-sectretase inhibitor IV from ...,B,None,None,BAO_0000190,...,Homo sapiens,Amyloid-beta precursor protein,9606,None,None,IC50,nM,UO_0000065,NaN,2100.0
2067,None,1074592,28446188,[],CHEMBL5737142,BiaCore assay: β-sectretase inhibitor IV from ...,B,None,None,BAO_0000190,...,Homo sapiens,Amyloid-beta precursor protein,9606,None,None,IC50,nM,UO_0000065,NaN,5000.0
2068,None,1074593,28446191,[],CHEMBL5737142,BiaCore assay: β-sectretase inhibitor IV from ...,B,None,None,BAO_0000190,...,Homo sapiens,Amyloid-beta precursor protein,9606,None,None,IC50,nM,UO_0000065,NaN,150.0


Data pre-processing

In [10]:
# Label compounds as either active, inactive, or intermediate
bioactivity_class = []
for i in df2.standard_value:
    if float(i) >= 10000:
        bioactivity_class.append("inactive")
    elif float(i) <= 1000:
        bioactivity_class.append("active")
    else:
        bioactivity_class.append("intermediate")

In [11]:
df3 = df2[["molecule_chembl_id", "canonical_smiles", "standard_value"]]
df4 = df3.join(pd.Series(bioactivity_class, name="bioactivity_class"), how='inner')
df4

,molecule_chembl_id,canonical_smiles,standard_value,bioactivity_class
0,CHEMBL311039,CC12CCC(C1)C(C)(C)C2NS(=O)(=O)c1ccc(F)cc1,5000.0,intermediate
1,CHEMBL450926,CC12CC[C@@H](C1)C(C)(C)[C@@H]2NS(=O)(=O)c1cccs1,2700.0,intermediate
2,CHEMBL310242,CC12CC[C@@H](C1)C(C)(C)[C@@H]2NS(=O)(=O)c1ccc(...,1800.0,intermediate
3,CHEMBL74874,CC12CC[C@@H](C1)C(C)(C)[C@@H]2NS(=O)(=O)c1ccc(...,11000.0,inactive
4,CHEMBL75183,CC12CC[C@@H](C1)C(C)(C)[C@@H]2NS(=O)(=O)c1ccc(...,10000.0,inactive
...,...,...,...,...
1843,CHEMBL3653692,CC1=NC2(N=C1N)c1cc(Br)ccc1CC21CCC(F)(F)CC1,399.0,active
1844,CHEMBL3653693,CC1=NC2(N=C1N)c1cc(-c3cncc(Cl)c3)ccc1CC21CCC(F...,53.0,intermediate
1845,CHEMBL3653693,CC1=NC2(N=C1N)c1cc(-c3cncc(Cl)c3)ccc1CC21CCC(F...,109.0,intermediate
1846,CHEMBL3653694,CC1=NC2(N=C1N)c1cc(NC(=O)c3ccc(Cl)cn3)ccc1CC21...,408.0,active


In [12]:
df4.to_csv('data/bioactivity_preprocessed_data.csv', index=False)